<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Block 1 — Load 19-column final dataset**

In [1]:
# ============================================================
# ML-05 / W05 — BLOCK 1
# LOAD FINAL 19-COLUMN FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/content/final_features_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("FINAL FEATURE DATASET")
print("=" * 70)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

print("\nShape:", df.shape)

FINAL FEATURE DATASET
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape: (2871202, 19)


**Block 2 — Missing indicators + 90-day eligibility**

In [2]:
# ============================================================
# BLOCK 2
# MISSING INDICATORS + BASIC DATA QUALITY
# ============================================================

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Remove meaningless missing indicator
# ------------------------------------------------------------

DROP_INDICATOR = "ai_other_missing"

if DROP_INDICATOR in df.columns:
    df = df.drop(columns=[DROP_INDICATOR])

print("=" * 70)
print("MISSING INDICATOR DECISION")
print("=" * 70)

print("Dropped:", DROP_INDICATOR)

remaining_indicators = [
    c for c in df.columns
    if c.endswith("_missing")
]

print("\nMissing indicators kept:")
for c in remaining_indicators:
    print("KEEP ->", c)

# ------------------------------------------------------------
# 2. Sort page-wise chronologically
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Check page history
# ------------------------------------------------------------

page_history = (
    df
    .groupby("content_hash_id")["month"]
    .nunique()
)

print("\n" + "=" * 70)
print("PAGE HISTORY")
print("=" * 70)

print(
    "Total pages:",
    f"{len(page_history):,}"
)

print(
    "Pages with < 3 months:",
    f"{(page_history < 3).sum():,}"
)

print(
    "Pages with >= 3 months:",
    f"{(page_history >= 3).sum():,}"
)

# ------------------------------------------------------------
# 4. Check monthly continuity
# ------------------------------------------------------------

df["previous_month"] = (
    df
    .groupby("content_hash_id")["month"]
    .shift(1)
)

df["month_gap"] = (
    (
        df["month"].dt.year
        - df["previous_month"].dt.year
    ) * 12
    +
    (
        df["month"].dt.month
        - df["previous_month"].dt.month
    )
)

gap_rows = df[
    df["month_gap"].notna()
    &
    (df["month_gap"] > 1)
]

print("\n" + "=" * 70)
print("MONTHLY CONTINUITY")
print("=" * 70)

print(
    "Rows with a month gap:",
    f"{len(gap_rows):,}"
)

print(
    "Pages affected:",
    f"{gap_rows['content_hash_id'].nunique():,}"
)

# helper columns no longer needed
df = df.drop(
    columns=[
        "previous_month",
        "month_gap"
    ]
)

MISSING INDICATOR DECISION
Dropped: ai_other_missing

Missing indicators kept:
KEEP -> gsc_avg_position_missing
KEEP -> ga4_total_engagement_sec_missing
KEEP -> sessions_organic_missing
KEEP -> sessions_ai_missing

PAGE HISTORY
Total pages: 427,292
Pages with < 3 months: 47,141
Pages with >= 3 months: 380,151

MONTHLY CONTINUITY
Rows with a month gap: 13,014
Pages affected: 11,525


**Block 3 — Feature impact + leakage checks**

In [3]:
# ============================================================
# BLOCK 3
# FEATURE AUDIT + LEAKAGE + VIF
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. ID columns are NOT model features
# ------------------------------------------------------------

ID_COLUMNS = [
    "client_hash_id",
    "content_hash_id"
]

# ------------------------------------------------------------
# 2. Potential future/label-derived names
# ------------------------------------------------------------

future_keywords = [
    "future",
    "target",
    "label",
    "decay",
    "next_",
    "lead_"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        suspicious_columns.append(col)

print("=" * 70)
print("LEAKAGE NAME CHECK")
print("=" * 70)

if suspicious_columns:
    print("Potentially suspicious columns:")
    for col in suspicious_columns:
        print("CHECK ->", col)
else:
    print("PASS: No obvious future/target-derived columns.")

# ------------------------------------------------------------
# 3. Constant columns
# ------------------------------------------------------------

feature_candidates = [
    c for c in df.columns
    if c not in ID_COLUMNS + ["month"]
]

constant_columns = [
    c for c in feature_candidates
    if df[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("ZERO-VARIANCE CHECK")
print("=" * 70)

if constant_columns:
    for c in constant_columns:
        print("DROP ->", c)
else:
    print("PASS: No zero-variance columns.")

# ------------------------------------------------------------
# 4. Numeric feature list
# ------------------------------------------------------------

numeric_features = [
    c for c in feature_candidates
    if c not in constant_columns
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nNumeric model candidates:")
for c in numeric_features:
    print(" -", c)

# ------------------------------------------------------------
# 5. Correlation between current features
# ------------------------------------------------------------

corr_matrix = (
    df[numeric_features]
    .corr()
)

print("\n" + "=" * 70)
print("HIGH FEATURE-TO-FEATURE CORRELATION")
print("=" * 70)

high_corr_pairs = []

for i in range(len(numeric_features)):

    for j in range(i + 1, len(numeric_features)):

        a = numeric_features[i]
        b = numeric_features[j]

        corr = corr_matrix.loc[a, b]

        if abs(corr) >= 0.90:

            high_corr_pairs.append(
                (a, b, round(corr, 3))
            )

if high_corr_pairs:

    for pair in high_corr_pairs:
        print(pair)

else:

    print("No feature pairs with |correlation| >= 0.90")

# ------------------------------------------------------------
# 6. VIF sample
# ------------------------------------------------------------
# VIF on millions of rows is unnecessarily expensive.
# We use a reproducible sample for diagnostic purposes.

vif_sample = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sample(
        n=min(100_000, len(df)),
        random_state=42
    )
)

# Remove columns with zero variance inside sample
vif_features = [
    c for c in numeric_features
    if vif_sample[c].nunique() > 1
]

X_vif = vif_sample[vif_features]

vif_table = pd.DataFrame({
    "feature": vif_features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            i
        )
        for i in range(X_vif.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("VIF TEST")
print("=" * 70)

display(vif_table)

LEAKAGE NAME CHECK
PASS: No obvious future/target-derived columns.

ZERO-VARIANCE CHECK
PASS: No zero-variance columns.

Numeric model candidates:
 - gsc_clicks
 - gsc_impressions
 - gsc_avg_position
 - ga4_total_engagement_sec
 - sessions_organic
 - sessions_ai
 - missing_count
 - gsc_avg_position_missing
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing
 - ctr
 - sec_per_click
 - ai_share
 - engagement_per_organic_session

HIGH FEATURE-TO-FEATURE CORRELATION
('gsc_clicks', 'sessions_organic', np.float64(0.978))
('missing_count', 'ga4_total_engagement_sec_missing', np.float64(0.968))
('missing_count', 'sessions_organic_missing', np.float64(0.968))
('missing_count', 'sessions_ai_missing', np.float64(0.968))
('ga4_total_engagement_sec_missing', 'sessions_organic_missing', np.float64(1.0))
('ga4_total_engagement_sec_missing', 'sessions_ai_missing', np.float64(1.0))
('sessions_organic_missing', 'sessions_ai_missing', np.float64(1.0))


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIF TEST


,feature,VIF
0,missing_count,inf
1,ga4_total_engagement_sec_missing,inf
2,gsc_avg_position_missing,inf
3,sessions_ai_missing,inf
4,sessions_organic_missing,inf
5,sessions_organic,4.518367
6,gsc_clicks,3.863913
7,gsc_impressions,2.184852
8,ga4_total_engagement_sec,2.176906
9,sessions_ai,1.924955


**Block 3.5: keep selected impactful  features after VIF and correlation test**

In [6]:
# ============================================================
# ML-05 → ML-06/07 PREPARATION
# FINAL FEATURE SET + 90-DAY ELIGIBILITY CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL FEATURE DATASET PREPARATION")
print("=" * 70)

# ============================================================
# 1. FIND THE CURRENT 19-COLUMN DATAFRAME
# ============================================================

# Try known dataframe names first
candidate_names = [
    "df_baseline",
    "df_model_base",
    "df_features_clean",
    "df_clean",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable dataframe found. "
        "Please load your 19-column Parquet dataset first."
    )

print(f"Source dataframe used: {source_name}")
print(f"Rows before cleaning: {len(source_df):,}")
print(f"Columns before cleaning: {source_df.shape[1]}")

# ============================================================
# 2. START FROM SOURCE DATA
# ============================================================

df_model_base = source_df.copy()

# Convert month safely
df_model_base["month"] = pd.to_datetime(
    df_model_base["month"],
    errors="coerce"
)

# ============================================================
# 3. REMOVE REDUNDANT MISSINGNESS FEATURES
# ============================================================

remove_columns = [
    "missing_count",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing"
]

# Only remove columns that actually exist
remove_columns = [
    col
    for col in remove_columns
    if col in df_model_base.columns
]

df_model_base = df_model_base.drop(
    columns=remove_columns
)

print("\n" + "=" * 70)
print("REMOVED REDUNDANT FEATURES")
print("=" * 70)

if remove_columns:
    for col in remove_columns:
        print(" -", col)
else:
    print("None")

# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "month"
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model_base.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

# ============================================================
# 5. REMOVE INVALID MONTH ROWS
# ============================================================

before_month_filter = len(df_model_base)

df_model_base = df_model_base[
    df_model_base["month"].notna()
].copy()

removed_invalid_months = (
    before_month_filter -
    len(df_model_base)
)

print(
    f"\nRows removed because month was invalid: "
    f"{removed_invalid_months:,}"
)

# ============================================================
# 6. REMOVE DUPLICATE PAGE-MONTH RECORDS IF ANY
# ============================================================

duplicate_count = (
    df_model_base
    .duplicated(
        subset=["content_hash_id", "month"]
    )
    .sum()
)

print(
    f"Duplicate page-month rows found: "
    f"{duplicate_count:,}"
)

if duplicate_count > 0:
    df_model_base = (
        df_model_base
        .drop_duplicates(
            subset=["content_hash_id", "month"],
            keep="first"
        )
        .copy()
    )

# ============================================================
# 7. SORT PAGE HISTORIES CHRONOLOGICALLY
# ============================================================

df_model_base = (
    df_model_base
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 8. CALCULATE OBSERVED HISTORY
# ============================================================

first_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("max")
)

df_model_base["content_age_days"] = (
    last_month - first_month
).dt.days

# ============================================================
# 9. 90-DAY ELIGIBILITY
# ============================================================

eligible_mask = (
    df_model_base["content_age_days"] >= 90
)

df_model_eligible = (
    df_model_base.loc[eligible_mask]
    .copy()
)

removed_rows = (
    len(df_model_base) -
    len(df_model_eligible)
)

print("\n" + "=" * 70)
print("90-DAY ELIGIBILITY")
print("=" * 70)

print(
    f"Rows before 90-day filter : "
    f"{len(df_model_base):,}"
)

print(
    f"Rows after 90-day filter  : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Rows removed              : "
    f"{removed_rows:,}"
)

# ============================================================
# 10. PAGE-LEVEL ELIGIBILITY CHECK
# ============================================================

page_age = (
    df_model_eligible
    .groupby("content_hash_id")["content_age_days"]
    .max()
)

if len(page_age) > 0:

    print(
        f"\nEligible pages: "
        f"{len(page_age):,}"
    )

    print(
        f"Minimum eligible history: "
        f"{page_age.min()} days"
    )

    print(
        f"Pages with <90 days remaining: "
        f"{(page_age < 90).sum():,}"
    )

    # Safety check
    assert (
        page_age >= 90
    ).all(), (
        "ERROR: A page below 90 days remains."
    )

else:
    raise ValueError(
        "No pages remain after the 90-day eligibility filter."
    )

# ============================================================
# 11. FINAL CHRONOLOGICAL ORDER
# ============================================================

df_model_eligible = (
    df_model_eligible
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 12. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Final rows    : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Final columns : "
    f"{df_model_eligible.shape[1]}"
)

print(
    f"Final shape   : "
    f"{df_model_eligible.shape}"
)

print("\nRemaining columns:")

for i, col in enumerate(
    df_model_eligible.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ============================================================
# 13. FINAL 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET — FIRST 5 ROWS")
print("=" * 70)

display(
    df_model_eligible.head(5)
)

# ============================================================
# 14. FINAL 90-DAY SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SAFETY CHECK")
print("=" * 70)

print(
    "Minimum content_age_days:",
    df_model_eligible["content_age_days"].min()
)

print(
    "Rows below 90 days:",
    (
        df_model_eligible["content_age_days"] < 90
    ).sum()
)

assert (
    df_model_eligible["content_age_days"] >= 90
).all()

print("PASS: No page below the 90-day requirement.")

# ============================================================
# 15. SAVE FINAL INTERMEDIATE DATASET
# ============================================================

output_path = (
    "/content/final_model_eligible.parquet"
)

df_model_eligible.to_parquet(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("PARQUET SAVED")
print("=" * 70)

print(output_path)

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FINAL FEATURE DATASET PREPARATION
Source dataframe used: df
Rows before cleaning: 2,871,202
Columns before cleaning: 18

REMOVED REDUNDANT FEATURES
 - missing_count
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing

Rows removed because month was invalid: 0
Duplicate page-month rows found: 0

90-DAY ELIGIBILITY
Rows before 90-day filter : 2,871,202
Rows after 90-day filter  : 2,705,303
Rows removed              : 165,899

Eligible pages: 349,557
Minimum eligible history: 92 days
Pages with <90 days remaining: 0

FINAL DATASET
Final rows    : 2,705,303
Final columns : 15
Final shape   : (2705303, 15)

Remaining columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. gsc_avg_position_missing
11. ctr
12. sec_per_click
13. ai_share
14. engagement_per_organic_session
15. content_age_days

FINAL DATASET — FIRST 5 ROWS


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,gsc_avg_position_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session,content_age_days
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0.006849,0.0,0.0,0.0,457
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457



FINAL SAFETY CHECK
Minimum content_age_days: 92
Rows below 90 days: 0
PASS: No page below the 90-day requirement.

PARQUET SAVED
/content/final_model_eligible.parquet

PREPARATION COMPLETE


**Block 4 — Rolling 90-day features + target**

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.